# 02_specialization — Entity specialization: one head field, N extension tables

A head row often continues in one of several tables, and which one depends on a value the row itself carries. An event is created, updated, deleted, moved or assigned, and each of those has its own table with its own columns and its own lifetime.

A relation as chapter 21 describes it cannot say this: a field's target type is exactly one, whichever container is chosen. Specialization declares the alternatives instead.


In [ ]:
!pip install aoa-action-machine


In [ ]:
from __future__ import annotations

from datetime import date
from typing import Annotated, get_args

from pydantic import Field
from typing_extensions import get_type_hints

from aoa.action_machine.domain import (
    BaseEntity,
    By,
    Generalization,
    Inverse,
    Rel,
    RelationNotLoadedError,
    Specialization,
    SpecializationDeclarationError,
    SpecializationOne,
    UndeclaredSpecializationVariantError,
)
from aoa.action_machine.domain.base_domain import BaseDomain
from aoa.action_machine.intents.entity import entity
from aoa.action_machine.runtime.action_product_machine import ActionProductMachine


## The head and its extensions

The head names its alternatives, each with the code that selects it, and the classifier field whose value chooses. `By(...)` names that field.

Every extension declares the way back and repeats its own code. The code stands on both sides on purpose: neither side is the authority, the match is.


In [ ]:
class EventsDomain(BaseDomain):
    name = "events"
    description = "Event log domain"


@entity(description="Event created", domain=EventsDomain)
class CreatedEventEntity(BaseEntity):
    id: str = Field(description="Same id as the event row")
    actor: str = Field(description="Who created the event")
    source: str = Field(description="Originating system")

    event: Annotated[
        Generalization(EventEntity, classifier="created_event"),
        Inverse(EventEntity, "details"),
    ] = Rel(description="Event this extension belongs to")

    origin_event: Annotated[
        Generalization(EventEntity, classifier="web"),
        Inverse(EventEntity, "origin"),
    ] = Rel(description="Event this extension belongs to, along the channel axis")


@entity(description="Event updated", domain=EventsDomain)
class UpdatedEventEntity(BaseEntity):
    id: str = Field(description="Same id as the event row")
    actor: str = Field(description="Who updated the event")
    changed_fields: str = Field(description="Comma-separated names of changed fields")

    event: Annotated[
        Generalization(EventEntity, classifier="updated_event"),
        Inverse(EventEntity, "details"),
    ] = Rel(description="Event this extension belongs to")

    origin_event: Annotated[
        Generalization(EventEntity, classifier="mobile"),
        Inverse(EventEntity, "origin"),
    ] = Rel(description="Event this extension belongs to, along the channel axis")


@entity(description="Event deleted", domain=EventsDomain)
class DeletedEventEntity(BaseEntity):
    id: str = Field(description="Same id as the event row")
    reason: str = Field(description="Why the event was deleted")
    soft: bool = Field(description="True when the row is only marked deleted")

    event: Annotated[
        Generalization(EventEntity, classifier="deleted_event"),
        Inverse(EventEntity, "details"),
    ] = Rel(description="Event this extension belongs to")

    origin_event: Annotated[
        Generalization(EventEntity, classifier="batch"),
        Inverse(EventEntity, "origin"),
    ] = Rel(description="Event this extension belongs to, along the channel axis")


@entity(description="Event moved", domain=EventsDomain)
class MovedEventEntity(BaseEntity):
    id: str = Field(description="Same id as the event row")
    from_stage: str = Field(description="Stage the event left")
    to_stage: str = Field(description="Stage the event entered")

    event: Annotated[
        Generalization(EventEntity, classifier="moved_event"),
        Inverse(EventEntity, "details"),
    ] = Rel(description="Event this extension belongs to")

    origin_event: Annotated[
        Generalization(EventEntity, classifier="web"),
        Inverse(EventEntity, "origin"),
    ] = Rel(description="Event this extension belongs to, along the channel axis")


@entity(description="Event assigned", domain=EventsDomain)
class AssignedEventEntity(BaseEntity):
    id: str = Field(description="Same id as the event row")
    assignee: str = Field(description="Who the event was assigned to")
    team: str = Field(description="Team the assignee belongs to")

    event: Annotated[
        Generalization(EventEntity, classifier="assigned_event"),
        Inverse(EventEntity, "details"),
    ] = Rel(description="Event this extension belongs to")

    origin_event: Annotated[
        Generalization(EventEntity, classifier="mobile"),
        Inverse(EventEntity, "origin"),
    ] = Rel(description="Event this extension belongs to, along the channel axis")


@entity(description="Event", domain=EventsDomain)
class EventEntity(BaseEntity):
    id: str = Field(description="Event id")
    event_date: date = Field(description="When the event happened")
    event_type: str = Field(description="Classifier: which extension table holds the details")
    channel: str = Field(description="Classifier: where the event came from")

    details: Annotated[
        Specialization(CreatedEventEntity, classifier="created_event"),
        Specialization(UpdatedEventEntity, classifier="updated_event"),
        Specialization(DeletedEventEntity, classifier="deleted_event"),
        Specialization(MovedEventEntity, classifier="moved_event"),
        Specialization(AssignedEventEntity, classifier="assigned_event"),
        Inverse(field_name="event"),
        By("event_type"),
    ] = Rel(description="Event details: one of the declared extensions")

    origin: Annotated[
        Specialization(CreatedEventEntity, classifier="web"),
        Specialization(UpdatedEventEntity, classifier="mobile"),
        Specialization(DeletedEventEntity, classifier="batch"),
        Specialization(MovedEventEntity, classifier="web"),
        Specialization(AssignedEventEntity, classifier="mobile"),
        Inverse(field_name="origin_event"),
        By("channel"),
    ] = Rel(description="Where the event came from: the same five extensions")


for _cls in (
    CreatedEventEntity,
    UpdatedEventEntity,
    DeletedEventEntity,
    MovedEventEntity,
    AssignedEventEntity,
    EventEntity,
):
    _cls.model_rebuild()


## 1. The model is the specification

Building the machine builds the graph, and the graph is where every rule of the declaration is checked: the codes on both sides match, every alternative carries a partner field, no class is claimed by two heads. A broken declaration raises here, at startup — not half a year later in a report.


In [ ]:
ActionProductMachine()
print("Machine built — the specialization declaration was accepted")


## 2. What the head's field says

Two axes over the same five extensions: `details` chosen by `event_type`, `origin` chosen by `channel`. Same classes, different classifiers and different codes.


In [ ]:
hints = get_type_hints(EventEntity, include_extras=True)

for field_name in EventEntity.model_fields:
    metadata = get_args(hints[field_name])[1:]
    alternatives = [(m.classifier, m.target_entity) for m in metadata if isinstance(m, Specialization)]
    if not alternatives:
        continue
    classifier = next(m.field_name for m in metadata if isinstance(m, By))
    print(f"{field_name}: chosen by `{classifier}`, {len(alternatives)} alternatives")
    for code_value, target in alternatives:
        print(f"   {code_value:<16} -> {target.__name__}")


## 3. The value the field holds

The id says which row, the variant says which table. With an un-hydrated row there is otherwise no way to tell which of the N tables holds the continuation — and reaching through an un-hydrated value fails explicitly instead of returning `None`.


In [ ]:
details = SpecializationOne(id="evt-1", variant="created_event", axis="details")
print(f"details.id = {details.id}   details.variant = {details.variant}   is_loaded={details.is_loaded}")

try:
    _ = details.actor
except RelationNotLoadedError as exc:
    print(f"details.actor -> {type(exc).__name__}: {exc}")


## 4. A code nobody declared

A missing continuation and an undeclared one are different facts about the data. Collapsing them into "no relation" hides a defect an operator needs to see, so the second is an explicit failure at the moment the value is built.


In [ ]:
try:
    SpecializationOne(id="evt-2", variant="archived_event", axis="details")
except UndeclaredSpecializationVariantError as exc:
    print(f"{type(exc).__name__}: {exc}")


## 5. The wrong table hydrated

The class of a hydrated entity must be the one declared for its code. Otherwise the row would be answered confidently and wrongly.


In [ ]:
try:
    SpecializationOne(
        id="evt-3",
        variant="created_event",
        axis="details",
        entity=DeletedEventEntity(id="evt-3", reason="oops", soft=True),
    )
except SpecializationDeclarationError as exc:
    print(f"{type(exc).__name__}: {exc}")


## 6. A declaration that disagrees with itself

The code stands on both sides so that the build can compare them, and this is the failure the feature exists to catch.

The broken model is declared with the good one above; it is meant to fail, so this notebook ends red on purpose. Two spellings are shown, and a reader who runs the notebook sees which one the framework names:

- the extension says `assigned` where its head says `assigned_event` — the codes disagree, and the build refuses to guess which side is right;
- a head field declared next to an ownership container — `Annotated[AssociationOne[X], Specialization(...), By(...)]` — which is refused rather than resolved to one of the two readings.

Remove the broken classes and the notebook describes a model that builds cleanly.


In [ ]:
@entity(description="Event assigned (a code that disagrees with the head)", domain=EventsDomain)
class BrokenAssignedEventEntity(BaseEntity):
    id: str = Field(description="Same id as the event row")
    assignee: str = Field(description="Who the event was assigned to")

    event: Annotated[
        Generalization(BrokenEventEntity, classifier="assigned"),  # the head says "assigned_event"
        Inverse(BrokenEventEntity, "details"),
    ] = Rel(description="Event this extension belongs to")


@entity(description="Event (the head of the broken pair)", domain=EventsDomain)
class BrokenEventEntity(BaseEntity):
    id: str = Field(description="Event id")
    event_type: str = Field(description="Classifier: which extension table holds the details")

    details: Annotated[
        Specialization(BrokenAssignedEventEntity, classifier="assigned_event"),
        Inverse(field_name="event"),
        By("event_type"),
    ] = Rel(description="Event details: one of the declared extensions")


BrokenAssignedEventEntity.model_rebuild()
BrokenEventEntity.model_rebuild()


In [ ]:
try:
    ActionProductMachine()
except SpecializationDeclarationError as exc:
    print(f"{type(exc).__name__}: {exc}")


The second spelling: a head field declared **beside** an ownership container — both readings are available and neither is right.


In [ ]:
@entity(description="Event (a head field declared beside an ownership container)", domain=EventsDomain)
class BrokenContainerHeadEntity(BaseEntity):
    id: str = Field(description="Event id")
    event_type: str = Field(description="Classifier: which extension table holds the details")

    # Both readings are available and neither is right: the container names one
    # target, the markers name five. The declaration is refused, not guessed.
    details: Annotated[
        AssociationOne[CreatedEventEntity],
        Specialization(CreatedEventEntity, classifier="created_event"),
        Inverse(field_name="event"),
        By("event_type"),
    ] = Rel(description="Ambiguous: a container and alternatives in one field")


BrokenContainerHeadEntity.model_rebuild()


In [ ]:
try:
    ActionProductMachine()
except Exception as exc:  # noqa: BLE001 — the reader is meant to see whatever the framework raises
    print(f"{type(exc).__name__}: {exc}")


## Summary

One head field, five extension tables, one classifier choosing among them. The declaration is checked at startup, the value carries both the id and the variant, and a row whose code nobody declared fails loudly instead of quietly returning nothing.

The last section is the reason this feature exists: a relation pointing at the wrong table is worse than a relation that fails, because nothing in a running system reports it.

Next — **[Lifecycle](../step_22-lifecycle.md)**: an entity's status as a finite-state machine with verifiable transitions.
